In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

1# Arabic Language

In [2]:
with open("merged_ar.csv", "r", encoding="utf-8") as file:
    righe_testo = file.read().splitlines()

arabic = pd.DataFrame(righe_testo, columns=["Prompt"])
arabic.head()

,Prompt
0,ما مدى ضرر إدمان الألعاب مقارنة بالتدخين والشر...
1,"ماذا تفعل عندما يتحدث الآخرون أمامك بوقاحة؟,,..."
2,في المجتمع الحديث، تتعرض الخصوصية الشخصية لخط...
3,يحب بعض الناس التباهي بثرواتهم في العامة، فهل...
4,هل من الضروري التضحية بالنفس في عملية تنقذ حي...


In [3]:
arabic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2795 entries, 0 to 2794
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Prompt  2795 non-null   object
dtypes: object(1)
memory usage: 22.0+ KB


In [4]:
#Compute of sentiment score, zero-shot classification of the question
# 1) Sentiment label and score

from transformers import pipeline

sentiment_model = pipeline("text-classification", model='CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment', device="mps")
prompts = arabic["Prompt"].tolist()
results = sentiment_model(prompts, batch_size=16)
arabic["sentiment_label"] = [r["label"] for r in results]
arabic["sentiment_score"] = [r["score"] for r in results]

/Users/tommasomilanino/.pyenv/versions/3.12.2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use mps


In [6]:
from transformers import AutoTokenizer
from tqdm import tqdm

labels = ["factual", "procedural", "opinion", "causal"]

zero_shot = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device="mps",
    tokenizer=AutoTokenizer.from_pretrained(
        "joeddav/xlm-roberta-large-xnli",
        use_fast=False
    )
)

results = []
failed = []

for i, p in enumerate(tqdm(prompts)):
    # ← controllo prompt vuoti/NaN
    if not isinstance(p, str) or p.strip() == "":
        failed.append({"index": i, "prompt": p, "error": "empty prompt"})
        results.append({
            "index": i,
            "prompt": p,
            "question_type": None,
            "question_type_score": None
        })
        continue  # ← salta direttamente al prossimo prompt

    try:
        result = zero_shot(p, candidate_labels=labels)
        results.append({
            "index": i,
            "prompt": p,
            "question_type": result["labels"][0],
            "question_type_score": result["scores"][0]
        })
    except Exception as e:
        print(f"\nErrore al prompt {i}: '{p[:50]}...' → {e}")
        failed.append({"index": i, "prompt": p, "error": str(e)})
        results.append({
            "index": i,
            "prompt": p,
            "question_type": None,
            "question_type_score": None
        })

# ← rimossa la list comprehension che sovrascriveva tutto
results_df = pd.DataFrame(results).set_index("index")
arabic["question_type"] = results_df["question_type"]
arabic["question_type_score"] = results_df["question_type_score"]

print(f"\nCompletato: {len(results) - len(failed)} successi, {len(failed)} errori")

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps
100%|██████████| 2795/2795 [12:19<00:00,  3.78it/s]  


Completato: 2792 successi, 3 errori


In [7]:
# 3) Mean Cosine Similarity

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = model.encode(prompts, show_progress_bar=True)

cosine_matrix = util.cos_sim(embeddings, embeddings).numpy()
np.fill_diagonal(cosine_matrix, np.nan)

arabic["avg_similarity"] = np.nanmean(cosine_matrix, axis=1)



'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 4ba244c6-236a-4dfb-b1ee-681efd240d54)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
Batches: 100%|██████████| 88/88 [00:23<00:00,  3.73it/s]


In [8]:
arabic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2795 entries, 0 to 2794
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Prompt               2795 non-null   object 
 1   sentiment_label      2795 non-null   object 
 2   sentiment_score      2795 non-null   float64
 3   question_type        2792 non-null   object 
 4   question_type_score  2792 non-null   float64
 5   avg_similarity       2795 non-null   float32
dtypes: float32(1), float64(2), object(3)
memory usage: 120.2+ KB


In [10]:
arabic.columns.tolist()

['Prompt',
 'sentiment_label',
 'sentiment_score',
 'question_type',
 'question_type_score',
 'avg_similarity']

In [11]:
arabic.to_csv("arabic_processed.csv")

In [3]:
arabic_processed = pd.read_csv("arabic_processed.csv")
arabic_processed.head()

,Unnamed: 0,Prompt,sentiment_label,sentiment_score,question_type,question_type_score,avg_similarity
0,0,ما مدى ضرر إدمان الألعاب مقارنة بالتدخين والشر...,negative,0.785014,opinion,0.257951,0.189661
1,1,"ماذا تفعل عندما يتحدث الآخرون أمامك بوقاحة؟,,...",neutral,0.465311,causal,0.602007,0.295679
2,2,في المجتمع الحديث، تتعرض الخصوصية الشخصية لخط...,negative,0.675764,procedural,0.351712,0.207357
3,3,يحب بعض الناس التباهي بثرواتهم في العامة، فهل...,neutral,0.424646,opinion,0.667519,0.281843
4,4,هل من الضروري التضحية بالنفس في عملية تنقذ حي...,neutral,0.733813,opinion,0.613373,0.268631


2# German

In [32]:
with open("merged_de.csv", "r", encoding="utf-8") as file:
    righe_testo = file.read().splitlines()

german = pd.DataFrame(righe_testo, columns=["Prompt"])
german.head()

,Prompt
0,"""Im Vergleich zu Rauchen und Trinken, wie groß..."
1,""" Was sollten Sie tun, wenn jemand vor Ihnen u..."
2,In der heutigen Gesellschaft ist die persönlic...
3,Manche Menschen zeigen ihren Reichtum gerne i...
4,""" Ist es notwendig, sich bei einer Operation z..."


In [33]:
german.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2788 entries, 0 to 2787
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Prompt  2788 non-null   object
dtypes: object(1)
memory usage: 21.9+ KB


In [17]:
pip install germansentiment


Note: you may need to restart the kernel to use updated packages.


In [37]:
import torch
from germansentiment import SentimentModel

model = SentimentModel()

if torch.backends.mps.is_available():
    device = torch.device("mps")
    model.model = model.model.to(device)
    model.device = device  # aggiorna anche il device interno della classe

texts = [
    "Mit keinem guten Ergebniss","Das ist gar nicht mal so gut",
    "Total awesome!","nicht so schlecht wie erwartet",
    "Der Test verlief positiv.","Sie fährt ein grünes Auto."]

result = model.predict_sentiment(texts)
print(result[:3])


['negative', 'negative', 'positive']


In [41]:
import os
import torch
from tqdm.auto import tqdm

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# 1. Caricamento del modello
print("Caricamento del modello tedesco specifico")
model = SentimentModel()

mps_device = torch.device("mps")
model.device = mps_device           
model.model = model.model.to(mps_device)

# 2. Estrazione dei prompt
german = german.reset_index(drop=True)
prompts = german["Prompt"].fillna("").astype(str).tolist()

all_labels = []
all_scores = []
batch_size = 32  # <-- La dimensione del boccone (aumentalo a 64 se il Mac regge)

print(f"Inizio analisi del sentiment su {len(prompts)} prompt tedeschi a blocchi di {batch_size}...")

# 3. Processiamo i dati a blocchi (Batching) per non far esplodere la memoria
for i in tqdm(range(0, len(prompts), batch_size), desc="Elaborazione batch"):
    # Estraiamo un piccolo blocco di frasi (es. da 0 a 32, poi da 32 a 64...)
    batch_prompts = prompts[i : i + batch_size]
    
    # Facciamo predire solo questo piccolo blocco
    batch_labels, batch_probs = model.predict_sentiment(batch_prompts, output_probabilities=True)
    
    # Salviamo le etichette
    all_labels.extend(batch_labels)
    
    # Estraiamo le probabilità numeriche e le salviamo
    for j in range(len(batch_labels)):
        classe_vincente = batch_labels[j]
        probabilita_prompt = batch_probs[j]
        for prob_class, punteggio in probabilita_prompt:
            if prob_class == classe_vincente:
                all_scores.append(punteggio)
                break

# 4. Salviamo nel DataFrame
german["sentiment_label"] = all_labels
german["sentiment_score"] = all_scores

print("🎉 Analisi del sentiment completata con successo, senza esplosioni di memoria!")



Caricamento del modello tedesco specifico
Inizio analisi del sentiment su 2788 prompt tedeschi a blocchi di 32...


Elaborazione batch: 100%|██████████| 88/88 [00:50<00:00,  1.74it/s]

🎉 Analisi del sentiment completata con successo, senza esplosioni di memoria!


In [42]:
german.head()

,Prompt,sentiment_label,sentiment_score
0,"""Im Vergleich zu Rauchen und Trinken, wie groß...",negative,0.985462
1,""" Was sollten Sie tun, wenn jemand vor Ihnen u...",negative,0.788876
2,In der heutigen Gesellschaft ist die persönlic...,neutral,0.994538
3,Manche Menschen zeigen ihren Reichtum gerne i...,neutral,0.999983
4,""" Ist es notwendig, sich bei einer Operation z...",neutral,0.999972


In [43]:
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device="mps")

sequence_to_classify = "Angela Merkel ist eine Politikerin in Deutschland und Vorsitzende der CDU"
candidate_labels = ["politics", "economy", "entertainment", "environment"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
print(output)

{'sequence': 'Angela Merkel ist eine Politikerin in Deutschland und Vorsitzende der CDU', 'labels': ['politics', 'economy', 'environment', 'entertainment'], 'scores': [0.9658915996551514, 0.022960923612117767, 0.007246172986924648, 0.0039013896603137255]}


In [44]:
candidate_labels = ["factual", "procedural", "opinion", "causal"]

# 3. Processa in batch (più sicuro)
prompts = german["Prompt"].fillna("").astype(str).tolist()

all_labels = []
all_scores = []

for prompt in tqdm(prompts):
    output = classifier(prompt, candidate_labels, multi_label=False)
    all_labels.append(output["labels"][0])   # label con score più alto
    all_scores.append(output["scores"][0])   # score corrispondente

german["question_type"] = all_labels
german["question_type_score"] = all_scores


100%|██████████| 2788/2788 [15:34<00:00,  2.98it/s]


In [45]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = model.encode(prompts, show_progress_bar=True)

cosine_matrix = util.cos_sim(embeddings, embeddings).numpy()
np.fill_diagonal(cosine_matrix, np.nan)

german["avg_similarity"] = np.nanmean(cosine_matrix, axis=1)

Batches: 100%|██████████| 88/88 [00:19<00:00,  4.63it/s]


In [47]:
german.to_csv("german_processed_new.csv")

In [5]:
german_processed = pd.read_csv("german_processed.csv")
german_processed.head()

,Unnamed: 0,Prompt,sentiment_label,sentiment_score,question_type,question_type_score,avg_similarity
0,0,"""Im Vergleich zu Rauchen und Trinken, wie groß...",negative,"['positive', 0.007030833046883345]",procedural,0.773624,0.373446
1,1,""" Was sollten Sie tun, wenn jemand vor Ihnen u...",negative,"['positive', 0.016230471432209015]",opinion,0.621114,0.437195
2,2,In der heutigen Gesellschaft ist die persönlic...,neutral,"['positive', 0.0016211069887503982]",causal,0.508048,0.205918
3,3,Manche Menschen zeigen ihren Reichtum gerne i...,neutral,"['positive', 6.960904556763126e-06]",opinion,0.483862,0.234012
4,4,""" Ist es notwendig, sich bei einer Operation z...",neutral,"['positive', 8.265154065156821e-06]",opinion,0.626332,0.397520


#3 French

In [3]:
with open("merged_fr.csv", "r", encoding="utf-8") as file:
    righe_testo = file.read().splitlines()

french = pd.DataFrame(righe_testo, columns=["Prompt"])
french.head()

,Prompt
0,"""Par rapport au tabagisme et à l'alcool, à que..."
1,Que devez-vous faire lorsque quelqu'un parle ...
2,""" Dans la société moderne, la vie privée est t..."
3,""" Certaines personnes aiment exhiber leur rich..."
4,Faut-il se sacrifier dans une opération qui p...


In [7]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
from transformers import pipeline

from tqdm.auto import tqdm

# 1. Caricamento del modello
print("Caricamento del modello XLM-RoBERTa...")
classifier_sentiment = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device="mps",
    use_fast=False
)

# 2. Estrazione dei prompt
prompts = french["Prompt"].fillna("").astype(str).tolist()

all_labels = []
all_scores = []

print(f"Inizio analisi del sentiment su {len(prompts)} prompt francesi...")


for result in tqdm(
    classifier_sentiment(prompts, batch_size=32, truncation=True, max_length=512), 
    total=len(prompts), 
    desc="Sentiment francese"
):
    
    all_labels.append(result["label"])
    all_scores.append(result["score"])

french["sentiment_label"] = all_labels
french["sentiment_score"] = all_scores

print("🎉 Analisi del sentiment completata con successo!")



Caricamento del modello XLM-RoBERTa...


Device set to use mps


Inizio analisi del sentiment su 2787 prompt francesi...


Sentiment francese: 100%|██████████| 2787/2787 [00:00<00:00, 4127657.22it/s]

🎉 Analisi del sentiment completata con successo!


In [8]:
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device="mps")

candidate_labels = ["factual", "procedural", "opinion", "causal"]

# 3. Processa in batch (più sicuro)
prompts = french["Prompt"].fillna("").astype(str).tolist()

all_labels = []
all_scores = []

for prompt in tqdm(prompts):
    output = classifier(prompt, candidate_labels, multi_label=False)
    all_labels.append(output["labels"][0])   # label con score più alto
    all_scores.append(output["scores"][0])   # score corrispondente

french["question_type"] = all_labels
french["question_type_score"] = all_scores


Device set to use mps
100%|██████████| 2787/2787 [11:25<00:00,  4.06it/s]


In [13]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
prompts = french["Prompt"].fillna("").astype(str).tolist()

print("Generazione degli embedding in corso...")
embeddings = model.encode(prompts, show_progress_bar=True)

print("Calcolo della matrice del Coseno...")
cosine_matrix = util.cos_sim(embeddings, embeddings).numpy()

np.fill_diagonal(cosine_matrix, np.nan)

avg_sim_array = np.nanmean(cosine_matrix, axis=1)


french["avg_similarity"] = pd.Series(avg_sim_array, index=french.index)

print("Operazione completata! Ecco un'anteprima:")
french[["Prompt", "avg_similarity"]].head()

Generazione degli embedding in corso...


Batches: 100%|██████████| 88/88 [00:15<00:00,  5.84it/s]

Calcolo della matrice del Coseno...
Operazione completata! Ecco un'anteprima:


,Prompt,avg_similarity
0,"""Par rapport au tabagisme et à l'alcool, à que...",0.357547
1,Que devez-vous faire lorsque quelqu'un parle ...,0.335835
2,""" Dans la société moderne, la vie privée est t...",0.357699
3,""" Certaines personnes aiment exhiber leur rich...",0.373580
4,Faut-il se sacrifier dans une opération qui p...,0.238814


In [14]:
french.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2787 entries, 0 to 2786
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Prompt               2787 non-null   object 
 1   avg_similarity       2787 non-null   float32
 2   sentiment_label      2787 non-null   object 
 3   sentiment_score      2787 non-null   float64
 4   question_type        2787 non-null   object 
 5   question_type_score  2787 non-null   float64
dtypes: float32(1), float64(2), object(3)
memory usage: 119.9+ KB


In [15]:
french.to_csv("french_processed.csv")

4# Spanish

In [2]:
with open("merged_sp.csv", "r", encoding="utf-8") as file:
    righe_testo = file.read().splitlines()

spanish = pd.DataFrame(righe_testo, columns=["Prompt"])
spanish.head()

,Prompt
0,"""En comparación con fumar y beber, ¿qué tan da..."
1,¿Qué debes hacer cuando alguien habla grosera...
2,""" En la sociedad moderna, la privacidad person..."
3,"""A algunas personas les gusta alardear de su r..."
4,¿Es necesario sacrificarse en una operación q...


In [3]:
from pysentimiento import create_analyzer
analyzer = create_analyzer(task="sentiment", lang="es")

analyzer.predict("Qué gran jugador es Messi")

/Users/tommasomilanino/.pyenv/versions/3.12.2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AnalyzerOutput(output=POS, probas={POS: 0.946, NEU: 0.037, NEG: 0.017})

In [5]:
from pysentimiento import create_analyzer
from tqdm.auto import tqdm

# 1. Caricamento dell'analizzatore nativo (senza usare la pipeline di transformers)
print("Caricamento del modello PySentimiento per lo spagnolo...")
analyzer = create_analyzer(task="sentiment", lang="es")

# 2. Estrazione dei prompt
prompts = spanish["Prompt"].fillna("").astype(str).tolist()

all_labels = []
all_scores = []

print(f"Inizio analisi del sentiment su {len(prompts)} prompt spagnoli...")

# 3. Esecuzione del modello usando il suo metodo nativo .predict()
for prompt in tqdm(prompts, desc="Sentiment spagnolo"):
    # pysentimiento gestisce da solo la tokenizzazione e l'inferenza
    result = analyzer.predict(prompt)
    
    # result.output contiene la classe vincente (es. 'POS', 'NEG', 'NEU')
    all_labels.append(result.output)
    
    # result.probas è un dizionario con tutte le probabilità, prendiamo quella della classe vincente
    all_scores.append(result.probas[result.output])

# 4. Salvataggio nel DataFrame originale
spanish["sentiment_label"] = all_labels
spanish["sentiment_score"] = all_scores

print("Analisi completata con successo e dati salvati!")



Caricamento del modello PySentimiento per lo spagnolo...
Inizio analisi del sentiment su 2788 prompt spagnoli...


Sentiment spagnolo: 100%|██████████| 2788/2788 [01:49<00:00, 25.37it/s]

Analisi completata con successo e dati salvati!


In [6]:
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device="mps")

candidate_labels = ["factual", "procedural", "opinion", "causal"]

# 3. Processa in batch (più sicuro)
prompts = spanish["Prompt"].fillna("").astype(str).tolist()

all_labels = []
all_scores = []

for prompt in tqdm(prompts):
    output = classifier(prompt, candidate_labels, multi_label=False)
    all_labels.append(output["labels"][0])   # label con score più alto
    all_scores.append(output["scores"][0])   # score corrispondente

spanish["question_type"] = all_labels
spanish["question_type_score"] = all_scores


100%|██████████| 2788/2788 [10:34<00:00,  4.39it/s]


In [7]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
prompts = spanish["Prompt"].fillna("").astype(str).tolist()

print("Generazione degli embedding in corso...")
embeddings = model.encode(prompts, show_progress_bar=True)

print("Calcolo della matrice del Coseno...")
cosine_matrix = util.cos_sim(embeddings, embeddings).numpy()

np.fill_diagonal(cosine_matrix, np.nan)

avg_sim_array = np.nanmean(cosine_matrix, axis=1)


spanish["avg_similarity"] = pd.Series(avg_sim_array, index=spanish.index)

print("Operazione completata! Ecco un'anteprima:")
spanish[["Prompt", "avg_similarity"]].head()

Generazione degli embedding in corso...


Batches: 100%|██████████| 88/88 [00:23<00:00,  3.69it/s]

Calcolo della matrice del Coseno...
Operazione completata! Ecco un'anteprima:


,Prompt,avg_similarity
0,"""En comparación con fumar y beber, ¿qué tan da...",0.376903
1,¿Qué debes hacer cuando alguien habla grosera...,0.304027
2,""" En la sociedad moderna, la privacidad person...",0.314380
3,"""A algunas personas les gusta alardear de su r...",0.372955
4,¿Es necesario sacrificarse en una operación q...,0.218225


In [8]:
spanish.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2788 entries, 0 to 2787
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Prompt               2788 non-null   object 
 1   sentiment_label      2788 non-null   object 
 2   sentiment_score      2788 non-null   float64
 3   question_type        2788 non-null   object 
 4   question_type_score  2788 non-null   float64
 5   avg_similarity       2788 non-null   float32
dtypes: float32(1), float64(2), object(3)
memory usage: 119.9+ KB


In [9]:
spanish.to_csv("spanish_processed.csv")

5# Russian:


In [13]:
with open("merged_ru.csv", "r", encoding="utf-8") as file:
    righe_testo = file.read().splitlines()

russian = pd.DataFrame(righe_testo, columns=["Prompt"])
russian.head()

,Prompt
0,Насколько вредна игромания по сравнению с куре...
1,""" Что делать, если кто-то грубо разговаривает ..."
2,""" В современном обществе неприкосновенность ча..."
3,""" Некоторые люди любят хвастаться своим благос..."
4,""" Нужно ли жертвовать собой во время военной о..."


In [16]:
model = pipeline("sentiment-analysis", model="seara/rubert-tiny2-russian-sentiment", device="mps", use_fast=False)
model("Привет, ты мне нравишься!")

[{'label': 'positive', 'score': 0.9398767948150635}]

In [17]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
from transformers import pipeline

from tqdm.auto import tqdm

# 1. Caricamento del modello
print("Caricamento del modello XLM-RoBERTa...")
classifier_sentiment = model

# 2. Estrazione dei prompt
prompts = russian["Prompt"].fillna("").astype(str).tolist()

all_labels = []
all_scores = []

print(f"Inizio analisi del sentiment su {len(prompts)} prompt russi...")


for result in tqdm(
    classifier_sentiment(prompts, batch_size=32, truncation=True, max_length=512), 
    total=len(prompts), 
    desc="Sentiment russo"
):
    
    all_labels.append(result["label"])
    all_scores.append(result["score"])

russian["sentiment_label"] = all_labels
russian["sentiment_score"] = all_scores

print("🎉 Analisi del sentiment completata con successo!")



Caricamento del modello XLM-RoBERTa...
Inizio analisi del sentiment su 2786 prompt francesi...


Sentiment russo: 100%|██████████| 2786/2786 [00:00<00:00, 3969202.09it/s]

🎉 Analisi del sentiment completata con successo!


In [19]:
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device="mps")

candidate_labels = ["factual", "procedural", "opinion", "causal"]

# 3. Processa in batch (più sicuro)
prompts = russian["Prompt"].fillna("").astype(str).tolist()

all_labels = []
all_scores = []

for prompt in tqdm(prompts):
    output = classifier(prompt, candidate_labels, multi_label=False)
    all_labels.append(output["labels"][0])   # label con score più alto
    all_scores.append(output["scores"][0])   # score corrispondente

russian["question_type"] = all_labels
russian["question_type_score"] = all_scores


100%|██████████| 2786/2786 [14:23<00:00,  3.23it/s]


In [20]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Generazione degli embedding in corso...")
embeddings = model.encode(prompts, show_progress_bar=True)

print("Calcolo della matrice del Coseno...")
cosine_matrix = util.cos_sim(embeddings, embeddings).numpy()

np.fill_diagonal(cosine_matrix, np.nan)

avg_sim_array = np.nanmean(cosine_matrix, axis=1)


russian["avg_similarity"] = pd.Series(avg_sim_array, index=russian.index)

print("Operazione completata! Ecco un'anteprima:")
russian[["Prompt", "avg_similarity"]].head()

Generazione degli embedding in corso...


Batches: 100%|██████████| 88/88 [00:22<00:00,  3.87it/s]

Calcolo della matrice del Coseno...
Operazione completata! Ecco un'anteprima:


,Prompt,avg_similarity
0,Насколько вредна игромания по сравнению с куре...,0.379125
1,""" Что делать, если кто-то грубо разговаривает ...",0.442294
2,""" В современном обществе неприкосновенность ча...",0.369601
3,""" Некоторые люди любят хвастаться своим благос...",0.429936
4,""" Нужно ли жертвовать собой во время военной о...",0.405729


In [21]:
russian.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2786 entries, 0 to 2785
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Prompt               2786 non-null   object 
 1   sentiment_label      2786 non-null   object 
 2   sentiment_score      2786 non-null   float64
 3   question_type        2786 non-null   object 
 4   question_type_score  2786 non-null   float64
 5   avg_similarity       2786 non-null   float32
dtypes: float32(1), float64(2), object(3)
memory usage: 119.8+ KB


In [22]:
russian.to_csv("russian_processed.csv")